In [ ]:
!pip install mysql-connector-python

In [2]:
import pandas as pd
import mysql.connector

In [3]:
conn = mysql.connector.connect(
    host = 'localhost',
    user = 'root',
    password = 'ahmedanwari',
    database = 'clinic'
)

In [4]:
query ="SELECT * FROM PATIENTS;"
df = pd.read_sql(query, conn)

In [5]:
df.to_excel("clinic_patients_file_updated.xlsx", index = False)
print("Data saved to excel file")


Data saved to excel file


Here is where you make edits to your file so that the code below has some rows to work with., if tehre are no rows then it will not have anythign to iterate over.

In [6]:
df = pd.read_excel("clinic_patients_file_updated.xlsx")

In [7]:
cursor = conn.cursor()

for i, row in df.iterrows():
    try:
        # Get values safely
        patient_id = row.get('PATIENT_ID')
        name = str(row['NAME'])
        problem = str(row['PROBLEM'])
        treatment = str(row['TREATMENT'])
        cost = float(row['COST'])
        payment = float(row['PAYMENT'])
        materials_used = str(row['MATERIALS_USED'])

        # Check if patient_id is missing or invalid
        if pd.isna(patient_id) or not str(patient_id).isdigit():
            # Insert new row
            cursor.execute("""
                INSERT INTO PATIENTS (NAME, PROBLEM, TREATMENT, COST, PAYMENT, MATERIALS_USED)
                VALUES (%s, %s, %s, %s, %s, %s)
            """, (name, problem, treatment, cost, payment, materials_used))
        else:
            # Check if patient id exists
            cursor.execute("SELECT COUNT(*) FROM PATIENTS WHERE PATIENT_ID = %s", (int(patient_id),))
            exists = cursor.fetchone()[0]

            if exists:
                # Update existing row
                cursor.execute("""
                    UPDATE PATIENTS SET
                        NAME = %s,
                        PROBLEM = %s,
                        TREATMENT = %s,
                        COST = %s,
                        PAYMENT = %s,
                        MATERIALS_USED = %s
                    WHERE PATIENT_ID = %s
                """, (name, problem, treatment, cost, payment, materials_used, int(patient_id)))
            else:
                # Insert with given ID
                cursor.execute("""
                    INSERT INTO PATIENTS (PATIENT_ID, NAME, PROBLEM, TREATMENT, COST, PAYMENT, MATERIALS_USED)
                    VALUES (%s, %s, %s, %s, %s, %s, %s)
                """, (int(patient_id), name, problem, treatment, cost, payment, materials_used))

    except Exception as e:
        print(f" Row {i} Failed: {e}")

conn.commit()
print("Database update complete.")


Database update complete.


In [8]:
df.head(20)



,PATIENT_ID,NAME,PROBLEM,TREATMENT,COST,PAYMENT,MATERIALS_USED
0,2,saad,Caries,filling,3000,3000,composite
1,3,Ali hamza,Caries,Cavity Filling,2500,2500,"Composite, Drill, Etchant"
2,4,Sarah Khan,Periodontitis,Deep Cleaning,3000,2000,"Scaling tools, Antibiotics"
3,5,Ayesha Patel,Pulpitis,Root Canal Therapy,8000,8000,"Gutta-percha, Sodium hypochlorite"
4,6,Arjun Mehra,Broken Tooth,Dental Crown,9000,6000,"Crown, Impression Paste"
5,7,Fatima Noor,Wisdom Tooth Pain,Extraction (surgical),7000,7000,"Forceps, Sutures, Gauze"
6,8,Bilal Zafar,Jaw Swelling,Antibiotics + Drainage,5000,3000,"Antibiotics, Syringes"
7,9,Riya Sharma,Teeth Alignment,Braces - Initial Setup,15000,10000,"Brackets, Wires, Elastics"
8,10,Ali Imran,Bleeding Gums,Gingival Curettage,4000,2000,"Curettes, Antiseptic"
9,11,NehaAmir,Tooth Sensitivity,Fluoride Treatment,2000,2000,"Fluoride Gel, Applicator"


In [9]:
cursor = conn.cursor()
cursor.execute("SELECT DATABASE();")
print("Active DB:", cursor.fetchone()[0])


Active DB: clinic


In [10]:
cursor.execute("SELECT * FROM PATIENTS;")
rows = cursor.fetchall()
for row in rows:
    print(row)


(2, 'saad', 'Caries', 'filling', 3000.0, 3000.0, 'composite')
(3, 'Ali hamza', 'Caries', 'Cavity Filling', 2500.0, 2500.0, 'Composite, Drill, Etchant')
(4, 'Sarah Khan', 'Periodontitis', 'Deep Cleaning', 3000.0, 2000.0, 'Scaling tools, Antibiotics')
(5, 'Ayesha Patel', 'Pulpitis', 'Root Canal Therapy', 8000.0, 8000.0, 'Gutta-percha, Sodium hypochlorite')
(6, 'Arjun Mehra', 'Broken Tooth', 'Dental Crown', 9000.0, 6000.0, 'Crown, Impression Paste')
(7, 'Fatima Noor', 'Wisdom Tooth Pain', 'Extraction (surgical)', 7000.0, 7000.0, 'Forceps, Sutures, Gauze')
(8, 'Bilal Zafar', 'Jaw Swelling', 'Antibiotics + Drainage', 5000.0, 3000.0, 'Antibiotics, Syringes')
(9, 'Riya Sharma', 'Teeth Alignment', 'Braces - Initial Setup', 15000.0, 10000.0, 'Brackets, Wires, Elastics')
(10, 'Ali Imran', 'Bleeding Gums', 'Gingival Curettage', 4000.0, 2000.0, 'Curettes, Antiseptic')
(11, 'NehaAmir', 'Tooth Sensitivity', 'Fluoride Treatment', 2000.0, 2000.0, 'Fluoride Gel, Applicator')
(12, 'Adnan Bashir', 'Fract